Qwen 2.5 Instruct Model: https://huggingface.co/Qwen/Qwen3.5-397B-A17B#benchmark-results:~:text=post%20Qwen3.5.-,Model%20Overview,-Type%3A%20Causal%20Language

In [ ]:
!pip -q install -U "transformers>=4.41.0" accelerate bitsandbytes sentencepiece pandas scikit-learn

import json, time
import pandas as pd
import numpy as np
import torch

from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix


Load Qwen2.5

In [ ]:
model_id = "Qwen/Qwen2.5-7B-Instruct"
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(
    model_id,
    use_fast=True,
    padding_side="left"
)

tokenizer.pad_token = tokenizer.eos_token


In [ ]:
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto",
    quantization_config=bnb_config,
)

print("Loaded:", model_id)

Load test set (kaggle)

In [ ]:
KAGGLE_CLEAN_PATH = "../kaggle_data.json"

with open(KAGGLE_CLEAN_PATH, "r", encoding="utf-8") as f:
    test_data = json.load(f)

X_test = [item["review_content"] for item in test_data]
y_test = [1 if item["is_positive"] else 0 for item in test_data]

print("Test samples (Kaggle):", len(X_test))
print("Pos:", sum(y_test), "Neg:", len(y_test)-sum(y_test))
print("Unique labels in y_test:", set(y_test))



In [ ]:
# Stratified Sampling:
# ensures unbiased slicing and balanced dataset
# takes exactly half pos and neg based on N size
import random
random.seed(42)

pos_idx = [i for i,y in enumerate(y_test) if y == 1]
neg_idx = [i for i,y in enumerate(y_test) if y == 0]

N = 2000  # take 2000 for now due to runtime lol
half = N // 2

idx = random.sample(pos_idx, half) + random.sample(neg_idx, N-half)
random.shuffle(idx)

X_sub = [X_test[i] for i in idx]
y_sub = [y_test[i] for i in idx]

print("Subset size:", len(X_sub))
print("Subset pos:", sum(y_sub), "neg:", len(y_sub)-sum(y_sub))
print("Unique labels:", set(y_sub))

Qwen classifier with strict label output to match our custom models

In [ ]:
import re

# helps parse LM output
def extract_label_from_generation(full_text: str):
    tail = full_text.split("Return:")[-1].strip().lower()
    m = re.search(r"\b(positive|negative)\b", tail)
    if m:
        return 1 if m.group(1) == "positive" else 0
    m2 = re.search(r"\b(positive|negative)\b", full_text.lower())
    if m2:
        return 1 if m2.group(1) == "positive" else 0
    return -1

def qwen_predict_labels(texts, batch_size=4, max_new_tokens=3, max_length=512):
    preds = []
    t0 = time.time()

    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        prompts = []

        for txt in batch:
            messages = [
                {"role": "system", "content": "You are a classifier. Return exactly one word: Positive or Negative. No other text."},
                {"role": "user", "content": f"{txt}\n\nReturn:"}
            ]
            prompts.append(tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True))

        inputs = tokenizer(
            prompts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=max_length
        ).to(model.device)

        with torch.no_grad():
            out = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=False,
                eos_token_id=tokenizer.eos_token_id,
            )

        decoded = tokenizer.batch_decode(out, skip_special_tokens=True)
        for full in decoded:
            preds.append(extract_label_from_generation(full))

    t1 = time.time()
    return preds, (t1 - t0)



In [ ]:
y_pred, seconds = qwen_predict_labels(X_sub, batch_size=16)

acc = accuracy_score(y_sub, y_pred)
prec = precision_score(y_sub, y_pred, zero_division=0)
rec = recall_score(y_sub, y_pred, zero_division=0)
f1 = f1_score(y_sub, y_pred, zero_division=0)

print("="*50)
print("RESULTS SUMMARY")
print("="*50)
print(f"Benchmark Model: Qwen with zero shot prompting")
print(f"Test data: Kaggle cleaned ({len(test_data):,} samples)")
print(f"\nMetrics:")
print(f"N:         {N}")
print(f"Seconds:   {seconds:.2f}  (sec/ex: {seconds/N:.4f})")
print(f"Accuracy:  {acc:.4f}")
print(f"Precision: {prec:.4f}")
print(f"Recall:    {rec:.4f}")
print(f"F1 Score:  {f1:.4f}")

print("\n--- Classification Report ---")
print(classification_report(y_sub, y_pred, target_names=["Negative","Positive"], zero_division=0))


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
cm = confusion_matrix(y_sub, y_pred)
print("\n--- Confusion Matrix ---")
print(cm)

# Plot confusion matrix
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Negative', 'Positive'],
            yticklabels=['Negative', 'Positive'])
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Benchmark Model Qwen - Confusion Matrix\n(Tested on Kaggle)')
plt.tight_layout()
plt.show()